# Stage 04: Data Acquisition and Ingestion

This notebook ingests recent AAPL daily prices from an API and the S&P 500 constituents table from a permitted public webpage. It parses data types, validates both datasets, and saves timestamped raw CSV files.

## Sources and Parameters

**API:** `https://www.alphavantage.co/query`, function `TIME_SERIES_DAILY`, symbol `AAPL`, output size `compact`. Authentication is loaded from `.env`; the key is never printed. If Alpha Vantage is unavailable or rate-limited, the assignment-approved fallback is `yfinance.download` with three months of daily AAPL observations.

**Scraping:** `https://en.wikipedia.org/wiki/List_of_S%26P_500_companies`, using the current constituents table selected by `table#constituents`.

In [1]:
import datetime as dt
import os
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data/raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(PROJECT_ROOT / ".env")
print("Project root:", PROJECT_ROOT)
print("Alpha Vantage key loaded:", bool(os.getenv("ALPHAVANTAGE_API_KEY")))

Project root: /Users/kevin/Desktop/NYU/bootcamp/bootcamp_zizhe_zhou/homework/stage4
Alpha Vantage key loaded: True


## Reusable Validation and Save Helpers

In [2]:
def timestamp() -> str:
    return dt.datetime.now().strftime("%Y%m%d-%H%M")


def validate_frame(df, required_columns, *, numeric_columns=None, date_columns=None):
    """Validate schema, shape, missing values, and requested dtypes."""
    numeric_columns = numeric_columns or []
    date_columns = date_columns or []
    missing_columns = [c for c in required_columns if c not in df.columns]
    assert not missing_columns, f"Missing required columns: {missing_columns}"
    assert not df.empty, "Dataset must contain at least one row"
    wrong_numeric = [c for c in numeric_columns if not pd.api.types.is_numeric_dtype(df[c])]
    wrong_dates = [c for c in date_columns if not pd.api.types.is_datetime64_any_dtype(df[c])]
    assert not wrong_numeric, f"Columns are not numeric: {wrong_numeric}"
    assert not wrong_dates, f"Columns are not datetime: {wrong_dates}"
    return {
        "shape": df.shape,
        "missing_required_columns": missing_columns,
        "na_counts": df[required_columns].isna().sum().to_dict(),
        "duplicate_rows": int(df.duplicated().sum()),
    }


def save_raw_csv(df, filename_prefix):
    path = RAW_DIR / f"{filename_prefix}_{timestamp()}.csv"
    df.to_csv(path, index=False)
    print("Saved:", path.relative_to(PROJECT_ROOT))
    return path

## Part 1 — API Pull: AAPL Daily Prices

Alpha Vantage may return HTTP 200 with an informational message instead of a time series. The code checks the response schema before using the data.

In [3]:
SYMBOL = "AAPL"
ALPHA_URL = "https://www.alphavantage.co/query"
api_key = os.getenv("ALPHAVANTAGE_API_KEY") or os.getenv("API_KEY")
api_source = "alpha_vantage"
alpha_payload = {}

if api_key:
    try:
        response = requests.get(ALPHA_URL, params={
            "function": "TIME_SERIES_DAILY", "symbol": SYMBOL,
            "outputsize": "compact", "apikey": api_key}, timeout=30)
        response.raise_for_status()
        alpha_payload = response.json()
    except (requests.RequestException, ValueError) as error:
        print(f"Alpha Vantage unavailable ({type(error).__name__}); using fallback.")
else:
    print("No API key found; using fallback.")

time_series = alpha_payload.get("Time Series (Daily)")
if time_series:
    api_df = (pd.DataFrame.from_dict(time_series, orient="index")
              .rename_axis("date").reset_index().rename(columns={
                  "1. open": "open", "2. high": "high",
                  "3. low": "low", "4. close": "close",
                  "5. volume": "volume"}))
else:
    if alpha_payload:
        print("Alpha Vantage returned no time series; using yfinance fallback.")
    import yfinance as yf
    api_source = "yfinance"
    api_df = yf.download(SYMBOL, period="3mo", interval="1d",
        auto_adjust=False, progress=False, multi_level_index=False).reset_index()
    api_df = api_df.rename(columns={column: column.lower() for column in api_df.columns})

api_columns = ["date", "open", "high", "low", "close", "volume"]
api_df = api_df[api_columns].copy()
api_df["date"] = pd.to_datetime(api_df["date"], errors="raise").dt.tz_localize(None)
for column in api_columns[1:]:
    api_df[column] = pd.to_numeric(api_df[column], errors="raise")
api_df = api_df.sort_values("date").reset_index(drop=True)
api_validation = validate_frame(api_df, api_columns, numeric_columns=api_columns[1:], date_columns=["date"])
assert (api_df[["open", "high", "low", "close"]] > 0).all().all()
assert (api_df["high"] >= api_df["low"]).all()
assert (api_df["volume"] >= 0).all()
print("API source used:", api_source)
print("Validation:", api_validation)
display(api_df.head())

API source used: alpha_vantage
Validation: {'shape': (100, 6), 'missing_required_columns': [], 'na_counts': {'date': 0, 'open': 0, 'high': 0, 'low': 0, 'close': 0, 'volume': 0}, 'duplicate_rows': 0}


,date,open,high,low,close,volume
0,2026-03-30,250.07,250.87,245.510,246.63,39446213
1,2026-03-31,247.91,255.48,247.101,253.79,49598091
2,2026-04-01,254.08,256.18,253.330,255.63,40059432
3,2026-04-02,254.20,256.13,250.650,255.92,31289369
4,2026-04-06,256.51,262.16,256.460,258.86,29329911


In [4]:
api_path = save_raw_csv(api_df, f"api_{api_source}_{SYMBOL}")

Saved: data/raw/api_alpha_vantage_AAPL_20260821-1152.csv


## Part 2 — Scrape the S&P 500 Constituents Table

The resilient selector targets the table's HTML id instead of assuming it is the first table on the page. Headers are mapped explicitly.

In [5]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
scrape_response = requests.get(SCRAPE_URL, headers={
    "User-Agent": "NYU-Stage04-Student-Project/1.0"}, timeout=30)
scrape_response.raise_for_status()
soup = BeautifulSoup(scrape_response.text, "html.parser")
table = soup.select_one("table#constituents")
assert table is not None, "Could not find table#constituents"
table_rows = table.select("tr")
headers = [cell.get_text(" ", strip=True) for cell in table_rows[0].select("th, td")]
rows = [[cell.get_text(" ", strip=True) for cell in row.select("th, td")]
        for row in table_rows[1:]]
rows = [row for row in rows if len(row) == len(headers)]
scrape_df = pd.DataFrame(rows, columns=headers).rename(columns={
    "Symbol": "symbol", "Security": "security",
    "GICS Sector": "gics_sector", "GICS Sub-Industry": "gics_sub_industry",
    "Headquarters Location": "headquarters_location",
    "Date added": "date_added", "CIK": "cik", "Founded": "founded"})
scrape_columns = ["symbol", "security", "gics_sector", "gics_sub_industry",
    "headquarters_location", "date_added", "cik", "founded"]
scrape_df = scrape_df[scrape_columns].copy()
scrape_df["date_added"] = pd.to_datetime(scrape_df["date_added"], errors="coerce")
scrape_df["cik"] = scrape_df["cik"].astype("string").str.zfill(10)
for column in ["symbol", "security", "gics_sector", "gics_sub_industry"]:
    scrape_df[column] = scrape_df[column].astype("string").str.strip()
scrape_validation = validate_frame(scrape_df, scrape_columns, date_columns=["date_added"])
assert len(scrape_df) >= 400, "Unexpectedly small constituents table"
assert scrape_df["symbol"].str.len().gt(0).all()
assert scrape_df["security"].str.len().gt(0).all()
assert scrape_df["symbol"].is_unique
print("Validation:", scrape_validation)
display(scrape_df.head())

Validation: {'shape': (503, 8), 'missing_required_columns': [], 'na_counts': {'symbol': 0, 'security': 0, 'gics_sector': 0, 'gics_sub_industry': 0, 'headquarters_location': 0, 'date_added': 0, 'cik': 0, 'founded': 0}, 'duplicate_rows': 0}


,symbol,security,gics_sector,gics_sub_industry,headquarters_location,date_added,cik,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,0000066740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee , Wisconsin",2017-07-26,0000091142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,0000001800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,0001551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin , Ireland",2011-07-06,0001467373,1989


In [6]:
scrape_path = save_raw_csv(scrape_df, "scrape_wikipedia_sp500_constituents")

Saved: data/raw/scrape_wikipedia_sp500_constituents_20260821-1152.csv


## Validation Logic

Both datasets must be non-empty and contain required columns. The notebook reports shape, NA counts, and duplicates, and validates numeric/date dtypes. API rows also require positive OHLC prices, `high >= low`, and non-negative volume. The scraped table requires at least 400 rows, non-empty company identifiers, and unique symbols.

## Assumptions and Risks

- Alpha Vantage may be rate-limited despite HTTP 200, so its JSON schema is checked and the actual source used is recorded.
- API schemas and adjusted-price behavior may change; timestamped raw files preserve acquisition snapshots.
- Wikipedia is community-maintained, and selectors or headers may change. Specific selectors and assertions fail loudly rather than saving the wrong table.
- S&P 500 membership changes over time; this scrape is a current snapshot, not membership history.
- Missing `date_added` values are allowed and reported because the source may omit them.
- This coursework workflow is not investment advice.

## Secret Hygiene

The API key remains only in local `.env`. Both repository and Stage 04 `.gitignore` rules exclude `.env`; `.env.example` contains dummy placeholders and is safe to commit.